# PerovStats main notebook
### This notebook is the main one to use when running PerovStats. It can take a directory containing multiple images and process them all in one run.

#### Before running this notebook please read the installation instructions (in the `/docs/` folder) to make sure the project dependencies are set up for the program.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import time

from loguru import logger
from pathlib import Path
import yaml
from topostats.io import LoadScans
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

sys.path.append(os.path.abspath(os.path.join('..')))
from src.perovstats.core.classes import PerovStats, ImageData
from src.perovstats.core.io import save_to_csv, save_config
from src.perovstats.cli import setup_logger
from src.perovstats.fourier import split_frequencies
from src.perovstats.segmentation import segment_image
from src.perovstats.processing import format_time, completion_message
from src.perovstats.smears import find_smear_areas
from src.perovstats.grains import find_grains
from src.perovstats.pruning import find_indents

---
#### These options can be changed to suit your needs. the `output_dir` folder will contain a selection of images from various stages in the process as well as the `.csv` files generated.

`img_files` should be a directory containing `.spm` files, not the actual file itself.

If you have a custom config file you would like to use also enter the path into the `config_path` variable. If not, the default config is already loaded in ready for you.

In [2]:
base_dir = Path("path/to/folder/with/.spm(s)")
output_dir = Path("folder/to/save/data/to")
config_path = Path("../src/perovstats/default_config.yaml")

base_dir = Path("../example images")
output_dir = Path("./output")
config_path = Path("../src/perovstats/default_config.yaml")

#### Select the segmentation method to use
#### As a reminder:
- `traditional`:
    - hard-coded
    - requires configuration options
    - faster (a few seconds/image)
    - will contain inaccuracies
- `cellpose`:
    - machine learning
    - no manual configuration required
    - slower (a few minutes/image) - unless your device uses an Nvidia GPU
    - much more accurate

In [3]:
segmentation_method = "cellpose" # Options: traditional, cellpose

---
#### Here the config and images will be loaded in ready for the main process

In [4]:
setup_logger()

with config_path.open() as f:
    config = yaml.safe_load(f)

config["segmentation"]["segmentation_type"] = segmentation_method

file_ext = config["file_ext"]
img_files = list(Path(base_dir).glob("*" + file_ext))

time_start = time.perf_counter()

# Load scans
load_config = config["loading"]
loadscans = LoadScans(img_files, config)
try:
    loadscans.get_data()
except ValueError as e:
    logger.warning(e)
    logger.warning(f"Channel {load_config['channel']} not found in file. Please ensure the config option is correct and all files contain the required channel.")
image_dicts = loadscans.img_dict

# Create the dataclasses for the whole process and for each image found
perovstats_object = PerovStats(config=config, images=[])
for filename, topostats_object in image_dicts.items():
    image_data = ImageData(
        success=True,
        filename=filename,
        pixel_to_nm_scaling=topostats_object.pixel_to_nm_scaling,
        image_original=topostats_object.image_original,
        image_flattened=None)
    perovstats_object.images.append(image_data)

logger.info("----------------------------------------------------------")
logger.info(f"Loaded {len(perovstats_object.images)} images")


[Thu, 02 Apr 2026 11:33:28] [INFO    ] [topostats] Extracting image from ..\example images\4_c60_perovonsil_ref_10um.PFQNM.spm
11:33:28 | INFO    | spm.py          | Loading image from : ..\example images\4_c60_perovonsil_ref_10um.PFQNM.spm
11:33:28 | INFO    | spm.py          | [4_c60_perovonsil_ref_10um.PFQNM] : Loaded image from : ..\example images\4_c60_perovonsil_ref_10um.PFQNM.spm
11:33:28 | INFO    | spm.py          | [4_c60_perovonsil_ref_10um.PFQNM] : Extracted channel Height
11:33:28 | INFO    | spm.py          | [4_c60_perovonsil_ref_10um.PFQNM] : Pixel to nm scaling : 19.53125
[Thu, 02 Apr 2026 11:33:28] [INFO    ] [topostats] Extracting image from ..\example images\TR_5um_ST4_14_26_per_Si_tapp4.0_00010.spm
11:33:28 | INFO    | spm.py          | Loading image from : ..\example images\TR_5um_ST4_14_26_per_Si_tapp4.0_00010.spm
11:33:28 | INFO    | spm.py          | [TR_5um_ST4_14_26_per_Si_tapp4.0_00010] : Loaded image from : ..\example images\TR_5um_ST4_14_26_per_Si_tapp4.0_

---
#### This is the main PerovStats process. Here images will be cycled through and processed one by one. You can see updates on its progress from the logs appearing under the code block

When mask segmentation completes for an image, warnings may appear. As long as the screen `SUCCESS` log appears under it, you can ignore these warnings.

In [ ]:
for idx, image_object in enumerate(perovstats_object.images):
    logger.info("----------------------------------------------------------")
    logger.info(f"Processing {image_object.filename} ({idx+1}/{len(perovstats_object.images)})")
    logger.info("----------------------------------------------------------")
    logger.debug(f"[{image_object.filename}] : Image dimensions: {image_object.image_original.shape}")
    logger.debug(f"[{image_object.filename}] : pixel_to_nm_scaling: {image_object.pixel_to_nm_scaling}")

    # Apply fourier transform to split the image into a low-passed and high-passed image
    split_frequencies(perovstats_object.config, image_object)

    # If frequency splitting was run and failed skip processing on the rest of the image
    if not image_object.success:
        continue

    # Generate segmented mask of the high-passed image
    segment_image(perovstats_object.config, image_object)

    # Find smear areas to be ignored/ removed
    find_smear_areas(perovstats_object.config, image_object)

    # Identify individual grains from mask and generate statistics on them
    find_grains(perovstats_object.config, image_object)

    # Find and mark offshoots in grains
    find_indents(perovstats_object.config, image_object)

    logger.info(f"[{image_object.filename}] : *** Exporting data ***")
    # Save image and grain data to their own .csv file
    image_df = pd.DataFrame([image_object.to_dict()])
    grains_list = []
    for grain in image_object.grains.values():
        grains_list.append(grain.to_dict())
    grain_df = pd.DataFrame(grains_list)

    output_filename = f"{output_dir}/{image_object.filename}/image_statistics.csv"
    save_to_csv(image_df, output_filename)
    output_filename = f"{output_dir}/{image_object.filename}/grain_statistics.csv"
    save_to_csv(grain_df, output_filename)
    # Save the config settings in a .yaml
    output_filename = Path(output_dir) / Path(image_object.filename) / "config.yaml"
    save_config(perovstats_object.config, output_filename)

    logger.info(
        f"[{image_object.filename}] : Exported data and config to {Path(output_dir) / Path(image_object.filename)}",
    )

11:33:29 | INFO    | 479166841.py    | ----------------------------------------------------------
11:33:29 | INFO    | 479166841.py    | Processing 4_c60_perovonsil_ref_10um.PFQNM.spm (1/2)
11:33:29 | INFO    | 479166841.py    | ----------------------------------------------------------
11:33:29 | DEBUG   | 479166841.py    | [4_c60_perovonsil_ref_10um.PFQNM.spm] : Image dimensions: (512, 512)
11:33:29 | DEBUG   | 479166841.py    | [4_c60_perovonsil_ref_10um.PFQNM.spm] : pixel_to_nm_scaling: 19.53125
11:33:29 | INFO    | fourier.py      | [4_c60_perovonsil_ref_10um.PFQNM.spm] : *** Frequency splitting ***
11:33:29 | INFO    | fourier.py      | [4_c60_perovonsil_ref_10um.PFQNM.spm] : Frequency cutoff: 0.1295 (301.6591nm)
11:33:29 | INFO    | fourier.py      | [4_c60_perovonsil_ref_10um.PFQNM.spm] : Splitting image frequencies
11:33:30 | INFO    | segmentation.py | [4_c60_perovonsil_ref_10um.PFQNM.spm] : *** Mask creation ***
11:33:30 | INFO    | segmentation.py | [4_c60_perovonsil_ref_10

c:\Users\tobya\Documents\Work\PerovStats\.venv\Lib\site-packages\cellpose\dynamics.py:524: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:767.)
  coo = torch.sparse_coo_tensor(pt, torch.ones(pt.shape[1], device=pt.device, dtype=torch.int),
Resizing is deprecated in v4.0.1+


11:37:16 | SUCCESS | segmentation.py | [4_c60_perovonsil_ref_10um.PFQNM.spm] : Mask created, Returning image to original size.
11:37:17 | INFO    | smears.py       | [4_c60_perovonsil_ref_10um.PFQNM.spm] : *** Finding smear areas ***
11:37:18 | INFO    | smears.py       | [4_c60_perovonsil_ref_10um.PFQNM.spm] : Smear areas found: 21 (11.7% of mask)
11:37:18 | INFO    | grains.py       | [4_c60_perovonsil_ref_10um.PFQNM.spm] : *** Grain finding ***
11:37:19 | INFO    | grains.py       | [4_c60_perovonsil_ref_10um.PFQNM.spm] : Obtained 688 grains
11:37:20 | INFO    | 479166841.py    | [4_c60_perovonsil_ref_10um.PFQNM.spm] : *** Exporting data ***
11:37:20 | INFO    | 479166841.py    | [4_c60_perovonsil_ref_10um.PFQNM.spm] : Exported data and config to output\4_c60_perovonsil_ref_10um.PFQNM.spm
11:37:20 | INFO    | 479166841.py    | ----------------------------------------------------------
11:37:20 | INFO    | 479166841.py    | Processing TR_5um_ST4_14_26_per_Si_tapp4.0_00010.spm (2/2)
1

Resizing is deprecated in v4.0.1+


11:41:08 | SUCCESS | segmentation.py | [TR_5um_ST4_14_26_per_Si_tapp4.0_00010.spm] : Mask created, Returning image to original size.
11:41:09 | INFO    | smears.py       | [TR_5um_ST4_14_26_per_Si_tapp4.0_00010.spm] : *** Finding smear areas ***
11:41:09 | INFO    | smears.py       | [TR_5um_ST4_14_26_per_Si_tapp4.0_00010.spm] : Smear areas found: 0 (0.0% of mask)
11:41:09 | INFO    | smears.py       | [TR_5um_ST4_14_26_per_Si_tapp4.0_00010.spm] : Minimum smear coverage not met, skipping smear removal.
11:41:09 | INFO    | grains.py       | [TR_5um_ST4_14_26_per_Si_tapp4.0_00010.spm] : *** Grain finding ***
11:41:10 | INFO    | grains.py       | [TR_5um_ST4_14_26_per_Si_tapp4.0_00010.spm] : Obtained 233 grains
11:41:11 | INFO    | 479166841.py    | [TR_5um_ST4_14_26_per_Si_tapp4.0_00010.spm] : *** Exporting data ***
11:41:11 | INFO    | 479166841.py    | [TR_5um_ST4_14_26_per_Si_tapp4.0_00010.spm] : Exported data and config to output\TR_5um_ST4_14_26_per_Si_tapp4.0_00010.spm


---
#### Once the process has been completed, basic stats about the run will be shown below.

Data about individual images will be saved in the `output_dir` you selected at the top with one subfolder for each image.

In [6]:
time_end = time.perf_counter()
time_taken = format_time(time_end - time_start)
time_per_image = format_time((time_end - time_start) / len(perovstats_object.images))
completion_message(perovstats_object, time_taken, time_per_image)

11:41:12 | SUCCESS | processing.py   | Process completed successfully.
----------------------------------------------------------------------------------------------------

 _______  _______  _______  _______           _______ _________ _______ _________ _______ 
(  ____ )(  ____ \(  ____ )(  ___  )|\     /|(  ____ \\__   __/(  ___  )\__   __/(  ____ \
| (    )|| (    \/| (    )|| (   ) || )   ( || (    \/   ) (   | (   ) |   ) (   | (    \/
| (____)|| (__    | (____)|| |   | || |   | || (_____    | |   | (___) |   | |   | (_____ 
|  _____)|  __)   |     __)| |   | |( (   ) )(_____  )   | |   |  ___  |   | |   (_____  )
| (      | (      | (\ (   | |   | | \ \_/ /       ) |   | |   | (   ) |   | |         ) |
| )      | (____/\| ) \ \__| (___) |  \   /  /\____) |   | |   | )   ( |   | |   /\____) |
|/       (_______/|/   \__/(_______)   \_/   \_______)   )_(   |/     \|   )_(   \_______)
                                                                                          

-------